
# CRISP-DM Workshop: Business Understanding & Data Understanding

**วัตถุประสงค์ของ Notebook นี้**
- ฝึกกำหนดเป้าหมายทางธุรกิจ (Business Objectives) และแปลงเป็น Data Mining Objectives
- ลงมือทำ Data Understanding: สร้าง/สำรวจชุดข้อมูล ตรวจหา Missing, Outliers, Duplicates และสรุป Data Quality Report

> เหมาะสำหรับการสอนปฏิบัติการ (Lab) ~90–120 นาที



## 1) Business Understanding

ในส่วนนี้ผู้เรียนจะ:
1. นิยาม **Business Objectives** (เชิงกลยุทธ์)
2. แปลงเป็น **Data Mining Objectives** (เชิงข้อมูล/เทคนิค)
3. กำหนด **Success Metrics/Criteria** ที่วัดผลได้



### ✅ Template กำหนดโจทย์
กรอกในเซลล์ด้านล่าง (Markdown) ให้ครบทั้ง 3 ส่วน

- **Business Objective:** (เช่น เพิ่มยอดขาย 20% ภายใน 6 เดือน)
- **Data Mining Objective:** (เช่น ทำ Recommendation System / ทำนาย Churn / ทำ Segmentation)
- **Success Criteria (ธุรกิจ):** (เช่น ยอดขายเฉลี่ย/CTR/Conversion เพิ่มขึ้นเป็น %)
- **Success Metrics (เชิงเทคนิค):** (เช่น Accuracy/Precision/Recall/RMSE/MAE/ARI/ROC-AUC)



> ✍️ **นักศึกษา/ผู้เรียนกรอกที่นี่**

- **Business Objective:** …
- **Data Mining Objective:** …
- **Success Criteria (ธุรกิจ):** …
- **Success Metrics (เทคนิค):** …



### 🧭 ตัวอย่าง Mapping (อ้างอิง)
| Business Objective | Data Mining Objective | เทคนิค |
|---|---|---|
| เพิ่มยอดขายออนไลน์ | Recommendation System | Association Rules, Collaborative Filtering |
| ลด Churn | ทำนายลูกค้าที่เสี่ยงลาออก | Logistic Regression, Random Forest |
| ลดต้นทุนสต็อก | Forecast Demand | Time Series (ARIMA, Prophet/LSTM)* |
| เพิ่มประสิทธิภาพการตลาด | Segmentation | Clustering (K-means/DBSCAN) |
| ตรวจจับทุจริต | Fraud Detection | Anomaly Detection, Classification |

\* ใน lab นี้จะไม่ใช้ไลบรารีภายนอกที่ต้องอินเทอร์เน็ต



## 2) Data Understanding

ในส่วนนี้เราจะใช้ **ชุดข้อมูลจำลองยอดขาย 2 ปี** เพื่อสาธิต
- การสำรวจโครงสร้างข้อมูล (schema/metadata)
- ตรวจสอบ **Missing Values**, **Outliers**, **Duplicates**, **Inconsistency**
- สรุปเป็น **Data Quality Report**


In [1]:

# ตั้งค่าเริ่มต้น (ไม่ต้องแก้)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)
pd.set_option('display.max_rows', 20)
pd.set_option('display.width', 120)


ModuleNotFoundError: No module named 'matplotlib'

### 2.1 สร้างชุดข้อมูลจำลอง (ยอดขาย 2 ปี)

In [ ]:

# สร้างข้อมูลจำลองยอดขาย 2 ปี (~200k records)
n = 200_000
dates = pd.date_range('2023-01-01', '2024-12-31', freq='D')
order_dates = np.random.choice(dates, size=n, replace=True)

df = pd.DataFrame({
    'Order_ID': np.arange(1, n+1),
    'Customer_ID': np.random.randint(10000, 20000, size=n).astype(str),
    'Customer_Age': np.random.normal(35, 10, size=n).round().astype('float'),
    'Product_ID': np.random.randint(1000, 1200, size=n).astype(str),
    'Quantity': np.random.poisson(lam=3, size=n).astype(int) + 1,
    'Revenue': np.random.gamma(shape=2.0, scale=200.0, size=n).round(2),
    'Order_Date': pd.to_datetime(order_dates)
})

# แทรก Delivery_Date (โดยปกติ >= Order_Date ภายใน 1-10 วัน)
df['Delivery_Date'] = df['Order_Date'] + pd.to_timedelta(np.random.randint(1, 11, size=n), unit='D')

# แทรก Missing Values (เช่น Customer_Age 10%, Revenue 2%)
mask_age = np.random.rand(n) < 0.10
mask_rev = np.random.rand(n) < 0.02
df.loc[mask_age, 'Customer_Age'] = np.nan
df.loc[mask_rev, 'Revenue'] = np.nan

# แทรก Outliers (Quantity > 1000, Revenue > 1,000,000)
out_idx_q = np.random.choice(df.index, size=800, replace=False)
df.loc[out_idx_q, 'Quantity'] = 1000 + np.random.randint(1, 50, size=800)
out_idx_r = np.random.choice(df.index.difference(out_idx_q), size=1200, replace=False)
df.loc[out_idx_r, 'Revenue'] = 1_000_000 + np.random.randint(1, 50_000, size=1200)

# แทรก Duplicates (ซ้ำ Customer_ID ประมาณ 1%)
dup_idx = np.random.choice(df.index, size=2000, replace=False)
df.loc[dup_idx, 'Customer_ID'] = df.loc[dup_idx, 'Customer_ID'].iloc[0]

# แทรก Inconsistency: Delivery_Date < Order_Date (~300 แถว) และ Order_Date ในอนาคต (~50 แถว)
inc_idx_deliv = np.random.choice(df.index, size=300, replace=False)
df.loc[inc_idx_deliv, 'Delivery_Date'] = df.loc[inc_idx_deliv, 'Order_Date'] - pd.to_timedelta(np.random.randint(1, 5, size=300), unit='D')

future_idx = np.random.choice(df.index.difference(inc_idx_deliv), size=50, replace=False)
df.loc[future_idx, 'Order_Date'] = pd.Timestamp('2025-12-31')  # วันที่อนาคต

df.head()


### 2.2 สำรวจโครงสร้าง/สถิติพื้นฐานของข้อมูล

In [ ]:

print("Shape:", df.shape)
print("\nข้อมูลประเภทคอลัมน์:")
print(df.dtypes)

print("\nสถิติพื้นฐาน (numeric):")
display(df.describe())

print("\nตัวอย่างข้อมูล:")
display(df.sample(5, random_state=1))


### 2.3 ตรวจ Missing Values

In [ ]:

missing_counts = df.isna().sum()
missing_pct = (missing_counts / len(df) * 100).round(2)
missing_report = pd.DataFrame({'Missing Count': missing_counts, 'Missing %': missing_pct}).sort_values('Missing %', ascending=False)
display(missing_report)


### 2.4 ตรวจ Outliers (ใช้ IQR rule)

In [ ]:

def iqr_outlier_count(series: pd.Series):
    s = series.dropna()
    if s.empty:
        return 0
    q1, q3 = np.percentile(s, [25, 75])
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return int(((s < lower) | (s > upper)).sum())

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
outlier_counts = {col: iqr_outlier_count(df[col]) for col in numeric_cols}
outlier_report = pd.DataFrame({'Outliers (IQR count)': outlier_counts}).sort_values('Outliers (IQR count)', ascending=False)
display(outlier_report)


### 2.5 ตรวจ Duplicates

In [ ]:

# ตัวอย่าง: ตรวจ duplicates ใน Customer_ID และ Order_ID
dup_customer = df.duplicated(subset=['Customer_ID']).sum()
dup_order = df.duplicated(subset=['Order_ID']).sum()

dup_report = pd.DataFrame({
    'Key': ['Customer_ID', 'Order_ID'],
    'Duplicate Rows': [dup_customer, dup_order]
})
display(dup_report)


### 2.6 ตรวจ Inconsistency (ความไม่สอดคล้องของข้อมูล)

In [ ]:

# 1) Delivery_Date ต้อง >= Order_Date
inconsist_delivery = (df['Delivery_Date'] < df['Order_Date']).sum()

# 2) Order_Date ไม่ควรอยู่ "อนาคต" (สมมติวันนี้คือวันที่สอน/รัน)
today = pd.Timestamp.today().normalize()
inconsist_future = (df['Order_Date'] > today).sum()

pd.DataFrame({
    'Rule': ['Delivery_Date >= Order_Date', 'Order_Date <= Today'],
    'Violation Count': [inconsist_delivery, inconsist_future]
})


### 2.7 การสำรวจเชิงภาพ (Exploratory Plots)

In [ ]:

# Histogram: Revenue (ตัดค่ามากๆ ออกเพื่อดูรูปทรง)
rev_clip = df['Revenue'].clip(upper=df['Revenue'].quantile(0.99))
plt.figure()
rev_clip.hist(bins=50)
plt.title('Revenue (clipped at 99th percentile)')
plt.xlabel('Revenue')
plt.ylabel('Frequency')
plt.show()


In [ ]:

# Boxplot: Quantity
plt.figure()
df['Quantity'].plot(kind='box')
plt.title('Quantity Boxplot')
plt.show()


### 2.8 สร้าง **Data Quality Report** (สรุปเป็นตาราง)

In [ ]:

def data_quality_report(df: pd.DataFrame):
    rows = []
    for col in df.columns:
        dtype = str(df[col].dtype)
        miss = int(df[col].isna().sum())
        miss_pct = round(miss / len(df) * 100, 2)

        # outliers เฉพาะ numeric โดย IQR
        out = None
        if np.issubdtype(df[col].dtype, np.number):
            out = iqr_outlier_count(df[col])
        else:
            out = 'N/A'

        # duplicates (นับแถวที่ทำให้คอลัมน์นี้ซ้ำ โดยพิจารณาคอลัมน์เดียว)
        dup = int(df.duplicated(subset=[col]).sum())

        note = ''
        if col in ['Delivery_Date', 'Order_Date']:
            note = 'ตรวจ rule วันที่'
        rows.append([col, dtype, miss, f"{miss_pct} %", out, dup, note])

    rep = pd.DataFrame(rows, columns=['Field', 'Data Type', 'Missing Count', 'Missing %', 'Outliers (IQR)', 'Duplicate Rows (by col)', 'Note'])
    return rep

dq = data_quality_report(df)
display(dq)



## 3) แบบฝึกหัดเสริม: เตรียมข้อมูล (เริ่มต้น Data Preparation)

ให้ผู้เรียนทำตาม **TODO** ต่อไปนี้ในเซลล์โค้ดด้านล่าง:
1. เติมค่า Missing ของ `Customer_Age` ด้วย **ค่า median** ของคอลัมน์
2. ตัด Outliers ของ `Quantity` ให้เพดานอยู่ที่เปอร์เซ็นไทล์ **99th**
3. ลบแถวที่ `Delivery_Date < Order_Date`
4. แก้วันที่ `Order_Date` ที่อยู่ในอนาคตให้เป็น **today**

> ทำเสร็จแล้วให้รันฟังก์ชัน `data_quality_report` ซ้ำเพื่อดูผลลัพธ์หลังการแก้ไข


In [ ]:

# TODO: ลงมือทำที่นี่
df_clean = df.copy()

# 1) Impute Customer_Age (median)
# your code here

# 2) Cap Quantity at 99th percentile
# your code here

# 3) Remove rows with Delivery_Date < Order_Date
# your code here

# 4) Fix future Order_Date to today
# your code here

# ตรวจซ้ำ
dq_after = data_quality_report(df_clean)
display(dq_after.head(20))
